# 07 · What a vector database adds to NumPy
Level L2 · Part 2 · builds on lesson 06

## Goal
By the end of this lesson you can list four things that break in lesson 06's NumPy search engine
when it meets a real product (restarts, deletes, growth, sharing), and name the vector-database
feature that fixes each one: persistence, a collection of points with ids, an index, and a server.

## The idea
Lesson 06's engine works, but only while one Python process stays alive and nobody changes the
recipes. A real recipe app restarts, edits and deletes recipes, grows to millions of them, and
serves many users at once. A **vector database** is software that stores vectors with their data
and keeps them searchable through all of that. It is not a different kind of maths: the scoring is
the same `vectors @ query` you wrote in lesson 04. Think of a notebook of phone numbers versus a
phone's contacts app: the numbers are the same, but the app saves them, finds them fast and
syncs them to your other devices.

## Picture

```mermaid
flowchart LR
    subgraph N["Lesson 06: NumPy, in memory"]
        A["recipes (DataFrame)"]
        B["doc_emb (50, 384)"]
        C["names (list)"]
    end
    subgraph D["Vector database: one collection"]
        P["point = id + vector + payload<br/>saved on disk, searchable by an index"]
    end
    N -- "three structures kept in step by you" --> D
```

On the left, row *i* of three separate objects must mean the same recipe, and everything vanishes
when Python stops. On the right, each recipe is one **point** that the database stores and finds.

## Step by step
### Step 1 · Rebuild lesson 06's engine
Same files, same `knn` (lesson 04), same query cache (lesson 06). Every notebook runs on its own,
so the pieces are re-declared here in short form.

In [1]:
import numpy as np
import pandas as pd

recipes = pd.read_csv("../data/recipes_50.csv")
names = recipes["title"].tolist()
doc_emb = np.load("../data/recipe_embeddings_minilm.npz")["doc_emb"]
cache = np.load("../data/query_embeddings_minilm.npz")
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))

def knn(query, vectors, k):
    scores = vectors @ np.asarray(query, dtype=np.float32)     # unit vectors: dot == cosine
    rows = np.argsort(-scores, kind="stable")[:k]
    return rows, scores[rows]

def search(text, k=3):
    rows, scores = knn(query_cache[text], doc_emb, k)
    return [(names[i], round(float(s), 3)) for i, s in zip(rows, scores, strict=True)]

print(search("seafood"))

[('Grilled salmon', 0.502), ('Sushi rolls', 0.489), ('Fish tacos', 0.473)]


Grilled salmon, Sushi rolls and Fish tacos, as in lesson 06. (This `knn` sorts all scores instead
of using `argpartition`; at 50 rows the answer is identical.) Now we stress it the way a real app
would.

### Step 2 · Break 1: a restart forgets every change
The kitchen stops making Beef stew, so we remove it. Three objects must change together: the
table, the matrix and the list of names.

In [2]:
row = names.index("Beef stew")
recipes = recipes.drop(index=row).reset_index(drop=True)
doc_emb = np.delete(doc_emb, row, axis=0)          # builds a new 49-row array (copies all the rest)
names = recipes["title"].tolist()
print("removed row", row, "-> recipes:", len(recipes), " vectors:", doc_emb.shape, " names:", len(names))
print("top hit for 'quick vegetarian dinner' now:", search("quick vegetarian dinner", k=1))

removed row 7 -> recipes: 49  vectors: (49, 384)  names: 49
top hit for 'quick vegetarian dinner' now: [('Full English breakfast', 0.515)]


The three objects agree (49 each), and lesson 06's wrong top hit, Beef stew, is replaced by
Full English breakfast: still meat, still wrong (filters fix that, Step 5). Now "restart" the app: a new process starts by
loading the files from disk, exactly as Step 1 did.

In [3]:
restarted = pd.read_csv("../data/recipes_50.csv")
print("after restart:", len(restarted), "recipes; Beef stew back?", "Beef stew" in set(restarted["title"]))

after restart: 50 recipes; Beef stew back? True


The delete only ever lived in memory. To keep it we would have to rewrite the CSV *and* the `.npz`
file ourselves, and a crash between the two writes would leave them out of step. **Persistence**
means changes are saved by the store itself, safely, as they happen.

### Step 3 · Break 2: row numbers are not identities
After the delete, every recipe below row 7 moved up by one. Anything that remembered "recipe at row
7" (a favourites list, a log, a cache) now points at a different dish.

In [4]:
original = pd.read_csv("../data/recipes_50.csv")
for r in [6, 7, 8]:
    print(f"row {r}: before delete = {original['title'][r]:22} after delete = {recipes['title'][r]}")
print("the CSV's own id for 'French onion soup':",
      int(original.loc[original["title"] == "French onion soup", "id"].iloc[0]))

row 6: before delete = Chicken noodle soup    after delete = Chicken noodle soup
row 7: before delete = Beef stew              after delete = French onion soup
row 8: before delete = French onion soup      after delete = Red lentil dal
the CSV's own id for 'French onion soup': 9


Row 7 used to be Beef stew and is now French onion soup. The CSV already has a better handle: an
`id` column that never changes. A vector database stores every vector under such an **id**, with
its data (the **payload**) attached, so there is nothing to keep in step.

### Step 4 · Break 3: every query reads every vector
Brute force scores all N vectors for each query (lesson 04). Time it at 50 rows and at 100,000
random unit vectors of the same size (seeded, so the data is the same each run).

In [5]:
import time

rng = np.random.default_rng(7)
big = rng.standard_normal((100_000, 384), dtype=np.float32)
big /= np.linalg.norm(big, axis=1, keepdims=True)
q = query_cache["seafood"]

for label, vecs in [("50 recipes", doc_emb), ("100,000 vectors", big)]:
    runs = []
    for _ in range(20):
        t0 = time.perf_counter(); knn(q, vecs, 3); runs.append(time.perf_counter() - t0)
    print(f"{label:16} median {np.median(runs) * 1000:8.3f} ms   matrix {vecs.nbytes / 1e6:8.1f} MB")

50 recipes       median    0.008 ms   matrix      0.1 MB


100,000 vectors  median   24.227 ms   matrix    153.6 MB


Your exact times depend on your laptop, but the pattern holds: thousands of times more vectors
costs roughly thousands of times more work and memory per query. Both grow in a straight line
with N, so we can project:

In [6]:
bytes_per_vector = 384 * 4                          # 384 float32 numbers
for n in [100_000, 1_000_000, 100_000_000, 1_000_000_000]:
    print(f"{n:>13,} recipes -> {n * bytes_per_vector / 1e9:8.1f} GB of vectors in RAM, "
          f"every query reads all of it")

      100,000 recipes ->      0.2 GB of vectors in RAM, every query reads all of it
    1,000,000 recipes ->      1.5 GB of vectors in RAM, every query reads all of it
  100,000,000 recipes ->    153.6 GB of vectors in RAM, every query reads all of it
1,000,000,000 recipes ->   1536.0 GB of vectors in RAM, every query reads all of it


A billion vectors is about 1,536 GB: more RAM than one machine has, read in full for every query.
The fix is an **index**: an extra structure built once so a query only looks at a small part of
the data (Part 4 explains IVF and HNSW). Databases add compression and on-disk storage too.

### Step 5 · The same 50 recipes in a collection
Now a real vector database: Qdrant, through `qdrant-client` in **local mode**, which runs inside
Python and saves to a folder (no server, no download). A **collection** is a named set of points
that share one vector size and one distance. Lesson 08 explains those two settings; here we copy
what lesson 06 used: 384 numbers, cosine.

In [7]:
import shutil, warnings
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

DB_PATH = "../data/cache/qdrant_07"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)
client.create_collection(
    collection_name="recipes",
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)
print(client.get_collections())

collections=[CollectionDescription(name='recipes')]


One empty collection called `recipes`. Next, each recipe becomes a point: its CSV `id`, its
vector, and a payload with the facts lesson 06 kept in the DataFrame. Lesson 09 covers points in
detail.

In [8]:
original_emb = np.load("../data/recipe_embeddings_minilm.npz")["doc_emb"]
points = [
    models.PointStruct(
        id=int(r.id),
        vector=original_emb[i].tolist(),
        payload={"title": r.title, "cuisine": r.cuisine,
                 "vegetarian": bool(r.vegetarian), "minutes": int(r.minutes)},
    )
    for i, r in enumerate(original.itertuples())
]
print(client.upsert(collection_name="recipes", points=points))
print(client.count(collection_name="recipes"))

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
count=50


`COMPLETED` and `count=50`. Now search it and put the answer next to NumPy's, on all 50 recipes.

In [9]:
hits = client.query_points(collection_name="recipes", query=query_cache["seafood"].tolist(), limit=3)
rows, scores = knn(query_cache["seafood"], original_emb, 3)
print("Qdrant:", [(p.id, p.payload["title"], round(p.score, 3)) for p in hits.points])
print("NumPy: ", [(int(original["id"][i]), original["title"][i], round(float(s), 3))
                  for i, s in zip(rows, scores, strict=True)])

Qdrant: [(40, 'Grilled salmon', 0.502), (42, 'Sushi rolls', 0.489), (39, 'Fish tacos', 0.473)]
NumPy:  [(40, 'Grilled salmon', 0.502), (42, 'Sushi rolls', 0.489), (39, 'Fish tacos', 0.473)]


The same three recipes with the same scores: the database does the same maths. What changed is
everything around it. Each hit carries its own id and payload, so there is no `names[i]` lookup
to get wrong. Filters are part of the query too (lesson 11):

In [10]:
quick_veg = models.Filter(must=[
    models.FieldCondition(key="vegetarian", match=models.MatchValue(value=True)),
    models.FieldCondition(key="minutes", range=models.Range(lte=30)),
])
hits = client.query_points(collection_name="recipes", query=query_cache["quick vegetarian dinner"].tolist(),
                           query_filter=quick_veg, limit=3)
print([(p.payload["title"], p.payload["minutes"], round(p.score, 3)) for p in hits.points])

[('Guacamole', 10, 0.484), ('Mango salsa', 10, 0.455), ('Spicy tofu stir-fry', 20, 0.444)]


Guacamole, Mango salsa and Spicy tofu stir-fry: lesson 06's masked result, with no mask to build
and no row numbers to map back.

### Step 6 · Fix 1 and 2: delete by id, then restart
Delete Beef stew by its id (8). Then close the client and open a new one on the same folder, which
is what a restarted app would do.

In [11]:
client.delete(collection_name="recipes", points_selector=models.PointIdsList(points=[8]))
client.close()

client = QdrantClient(path=DB_PATH)                   # "restart": a fresh client, same folder
print(client.count(collection_name="recipes"))
print([(p.id, p.payload["title"]) for p in client.retrieve(collection_name="recipes", ids=[7, 8, 9])])

count=49
[(7, 'Chicken noodle soup'), (9, 'French onion soup')]


After the restart there are 49 points: the delete was saved. Asking for ids 7, 8 and 9 returns 7
and 9 only; id 9 is still French onion soup, because deleting one point never renumbers the
others.

### Step 7 · Fix 3: where the index lives
Ask the collection for its details.

In [12]:
info = client.get_collection(collection_name="recipes")
print("points:", info.points_count, "  indexed vectors:", info.indexed_vectors_count)
print("index settings (HNSW):", "m =", info.config.hnsw_config.m,
      " ef_construct =", info.config.hnsw_config.ef_construct)

points: 49   indexed vectors: 0
index settings (HNSW): m = 16  ef_construct = 100


The collection *has* index settings (HNSW, Part 4), but `indexed vectors: 0`. Local mode is for
learning and tests: it always scores every point, like our NumPy. The Qdrant server builds the
index once a collection is big enough (lesson 13 compares the modes). The table sums up all four
breaks; the fourth (sharing) is Common mistake 1 below.

| What breaks in NumPy | Seen in | Database feature | Course |
| --- | --- | --- | --- |
| Changes vanish on restart | Step 2 | persistence (saved by the store) | 09, 12, 13 |
| Row numbers shift; three objects to keep in step | Step 3 | collection of points: id + vector + payload | 08, 09 |
| Every query reads every vector; RAM grows with N | Step 4 | index, compression, on-disk storage | Part 4 |
| Only one process can use the data | Mistake 1 | server; later replication and sharding | 13, Part 6 |

## What just happened
- Step 2 deleted Beef stew from three in-memory objects, and reloading the files brought it back: NumPy has no persistence.
- Step 3 showed row 7 changing from Beef stew to French onion soup after the delete, while the CSV's `id` (9) stayed fixed.
- Step 4 measured brute force at 50 and 100,000 vectors and projected 1 billion vectors to about 1,536 GB, all read per query.
- Step 5 loaded the same 50 recipes into a Qdrant collection and got the same top 3 and scores as NumPy, plus filters in the query.
- Step 6 deleted id 8, restarted the client, and still had 49 points with ids unchanged.
- Step 7 showed HNSW settings with 0 indexed vectors: local mode searches exactly, the server indexes.

## Common mistakes

**1. Opening the same local folder from two clients.** A second notebook, a web app and a
background job all want the recipes. In local mode they cannot share a folder:

In [13]:
try:
    second_app = QdrantClient(path=DB_PATH)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

RuntimeError: Storage folder ../data/cache/qdrant_07 is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.


It prints `RuntimeError: Storage folder ../data/cache/qdrant_07 is already accessed by another
instance of Qdrant client. If you require concurrent access, use Qdrant server instead.` Local
mode is one process, like NumPy. Fix: in one process, reuse the one `client`; for many users, run
the Qdrant server and connect with `QdrantClient(url=...)` (lesson 13).

In [14]:
print("one shared client:", client.count(collection_name="recipes"))

one shared client: count=49


**2. Using a row number as a recipe's identity.** A favourites list saved row numbers before the
delete in Step 2. After it, the same numbers point at other recipes:

In [15]:
try:
    favourites = [7, 12]                                   # saved as row numbers on the original 50 rows
    print("meant:", [original["title"][r] for r in favourites], " now:", [names[r] for r in favourites])
    assert [names[r] for r in favourites] == [original["title"][r] for r in favourites], "favourites changed"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

meant: ['Beef stew', 'Vanilla ice cream']  now: ['French onion soup', 'Strawberry cheesecake']
AssertionError: favourites changed


It prints `AssertionError: favourites changed`: Beef stew became French onion soup and Vanilla
ice cream (row 12) became Strawberry cheesecake. Fix: save stable ids and look them up in the database.

In [16]:
favourite_ids = [int(original["id"][r]) for r in [7, 12]]          # ids 8 and 13
print([(p.id, p.payload["title"]) for p in client.retrieve(collection_name="recipes", ids=favourite_ids)])

[(13, 'Vanilla ice cream')]


Id 13 still returns its recipe. Id 8 returns nothing because Beef stew really was deleted: a
missing result, not a wrong one.

## Try it
1. **Filter in both.** Search "seafood" for recipes ready in 30 minutes or less, once with a
   NumPy mask on `original_emb` and once with a Qdrant `Range` filter. Do they agree?
2. **When does brute force get slow?** Time `knn` at 10,000, 100,000 and 300,000 random vectors.
   Roughly how many vectors would make one query take 100 ms on your laptop?

In [17]:
# your code here

Close the client so the folder is released (the next lesson starts its own).

In [18]:
client.close()
print("closed")

closed


## Key terms
- **vector database** — software that stores vectors with their ids and data, keeps them on disk, and answers similarity queries quickly while data changes.
- **collection** — a named set of points in a vector database that share one vector size and one distance metric.
- **point** — one stored item in a collection: an id, a vector and a payload.
- **payload** — the data stored with a point's vector (title, cuisine, vegetarian, minutes), used for display and filters.
- **index** — an extra structure built from the stored vectors so a query looks at a small part of them instead of all.
- **persistence** — changes are saved by the store itself and survive a restart.
- **local mode** — `qdrant-client` running inside Python with no server: one process, exact search, for learning and tests.